# Nature 2026: sources, sampling and coverage

**Time:** 45 minutes · **Prerequisites:** chapters 03–04 or basic Python tables. No private input or GIS installation is required.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/15_nature_data_and_sampling.ipynb)

**Reading the book?** It shows saved results. You can zoom and rotate Plotly figures here; Python-backed selectors, sliders and uploads require running the notebook in JupyterLite.

Wait for **Python (Pyodide) | Idle**, then choose **Run → Run All Cells**. Public results below reflect the bundled aggregate artifact; unavailable models are labeled. Fictional exercises remain explicitly separate from the study.

In [1]:
import sys
from pathlib import Path
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'plotly', 'ipywidgets'])
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from coastlab import show, land_traces
from naturelab import (msl_elevation, screening_summary, paired_comparison,
                       load_public, load_sources, coverage_frame, fictional_samples,
                       aggregate_explorer, sensitivity_controls, terrain_explorer)
public = load_public()
sources = load_sources()
print('Ready: Nature 2026 — public aggregates and explicitly fictional exercises.')
print('Current sampling status:', public['status'])

Ready: Nature 2026 — public aggregates and explicitly fictional exercises.
Current sampling status: partial


## Objectives: make the denominator reproducible

Distinguish a catalog from raster data, pin versions and rights, and explain every missing cell. Follow an authorized workbook through validation, sampling and aggregate publication.

The October 2026 catalog lists **FABDEM v1.0**, **DeltaDTM v1.1**, and **GLL-DTMv2** corrected to MSL. The paper calls DeltaDTM **v1**; the current catalog's **v1.1** must be named explicitly when used. GLL's approximately 1 km grid is a portfolio screening input, not a building survey. Inspect actual raster CRS and cell size rather than inferring them from a filename.

The original public GLL v2 release is not an interchangeable fallback: the Nature methods identify a prior geoid-conversion error and describe obtaining a preconverted EGM96 version from the authors. This workflow uses the corrected WUR product, not an ad hoc correction of the public original.

In [2]:
catalog = pd.DataFrame(sources['products'])
display(catalog[['id', 'dataset', 'version', 'datum', 'source_url', 'license']])
print('Catalog version:', sources['catalog']['version_doi'])
display(coverage_frame(public)[['model', 'status', 'total', 'available', 'unavailable', 'at_or_below']])
print('Null means unavailable, not zero. Threshold shown: <=1 m MSL.')

,id,dataset,version,datum,source_url,license
0,fabdem,FABDEM,1.0,MSL,https://doi.org/10.17887/WUR01-6L9D2X,CC-BY-NC-SA-4.0
1,deltadtm,DeltaDTM,1.1,MSL,https://doi.org/10.17887/WUR01-GX7S1B,CC-BY-NC-SA-4.0
2,gll,GLL-DTM,2,MSL,https://doi.org/10.17887/WUR01-XGN3PK,CC-BY-NC-SA-4.0


Catalog version: 10.5281/zenodo.23341921


,model,status,total,available,unavailable,at_or_below
0,GLL-DTM v2 (global coastal lowlands),partial,25873,798,25075,86
1,FABDEM v1.0 (five poster-city regions),partial,25873,694,25179,10
2,DeltaDTM v1.1 (five poster-city regions),partial,25873,692,25181,35


Null means unavailable, not zero. Threshold shown: <=1 m MSL.


## Inspect public terrain context

When available, the following city selector displays real raster-derived terrain in 2D and 3D. Its grid note identifies the source model and display sampling. These public geographic extracts are independent of the private facility table: no facility points are plotted. Null cells and heights ≤−7 m stay masked. The 3D aspect ratio exaggerates height; the paired map is the 2D alternative. A terrain surface alone has no hydraulic connectivity, defenses, flood extent or probability.

In [3]:
terrain = terrain_explorer()

## A local pipeline with an auditable handoff

1. Use an authorized local copy of the `Climate-Sites-w-coords` worksheet. Validate required columns, unique `uid` and 0/1 `PEPFARSUP`. Select supported rows once.
2. Audit all 24 RCP scenario/decade counts against the published baseline. Invalid categories stop processing; discrepancies remain explicit.
3. Keep invalid coordinates in the denominator. For valid WGS84 longitude/latitude, transform into each raster CRS. Sample the **containing pixel**, with no interpolation or extrapolation. Apply the raster band's scale/offset. At an internal pixel boundary, rasterio's row/column convention selects the lower/right cell.
4. For overlapping tiles, use the first finite unmasked value in manifest order. NoData can fall through to another tile. Heights ≤−7 m are excluded; their raw values and reason remain private. Missing rasters, outside coverage and invalid coordinates remain unavailable.
5. Write detailed samples/audit only to ignored `private-data/`. Publish model metadata, coverage and whole-portfolio aggregates. Hash sources and record the generation time.

This deterministic point sampler is independent of the paper's global analysis and does not replicate its bilinear coastal sampling, water masking or geoid construction. A point inside a coarse cell inherits that cell's uncertainty.

**Local terminal, optional; never run as a browser dependency:**

```bash
python -m pip install -r requirements-geospatial.txt
python scripts/analyze_nature.py private-data/sites.xlsx --manifest private-data/nature-rasters.json
python scripts/update_provenance.py
```

See the [source audit](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/source-audit.html) for the manifest template and column overrides. Raster acquisition is explicit and separate; a default lesson never downloads large grids.

In [4]:
edge = np.array([-8, -7, -6.9, 0, 1, 10, 10.1, np.nan])
checks = pd.DataFrame([screening_summary(edge, t) for t in [0, 1, 10]])
print('FICTIONAL boundary fixture; not sampled facilities')
display(checks)
assert checks.at_or_below.tolist() == [2, 3, 4]
assert (checks.available + checks.unavailable == checks.total).all()
assert (checks.excluded_le_minus7 == 2).all()
show(go.Figure(go.Bar(x=['<=0 m', '<=1 m', '<=10 m'], y=checks.at_or_below))
     .update_layout(title='FICTIONAL thresholds include equality; 3 of 8 rows unavailable',
                    yaxis_title='Valid rows at/below threshold'))

FICTIONAL boundary fixture; not sampled facilities


,total,available,unavailable,missing,excluded_le_minus7,threshold_m,offset_m,at_or_below,above,percent_of_available
0,8,5,3,1,2,0.0,0.0,2,3,40.0
1,8,5,3,1,2,1.0,0.0,3,2,60.0
2,8,5,3,1,2,10.0,0.0,4,1,80.0


<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="a57d5c9c-9ad8-4b09-a03b-ca0e5e5456f6" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("a57d5c9c-9ad8-4b09-a03b-ca0e5e5456f6")) { Plotly.newPlot( "a57d5c9c-9ad8-4b09-a03b-ca0e5e5456f6", [{"x":["\u003c=0 m","\u003c=1 m","\u003c=10 m"],"y":{"dtype":"i1","bdata":"AgME"},"type":"bar"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"#E5ECF6","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"scattergeo":[{"type":"scattergeo","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolar":[{"type":"scatterpolar","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"histogram":[{"marker":{"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"histogram"}],"scattergl":[{"type":"scattergl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatter3d":[{"type":"scatter3d","line":{"colorbar":{"outlinewidth":0,"ticks":""}},"marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scattermap":[{"type":"scattermap","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterternary":[{"type":"scatterternary","marker

## Lab: missingness is a result

A model has eight portfolio rows: two are excluded at ≤−7 m, one is NoData and five are valid. Four valid heights are ≤10 m. Compute the share among available rows and the share of the full cohort known to be ≤10 m. Can the other three be described as dry?

**Worked answer:** 4/5=80% among available rows, while 4/8=50% of the full cohort are known to satisfy the threshold. Three rows are unavailable, so neither 50% nor 80% is a complete-cohort flood probability. If all cells are unavailable, the count is reported as unavailable, not zero.

**Deliverable:** A coverage ledger conserving `available + unavailable = total`, with reasons for unavailable rows, exact raster versions, interpolation choice and source hashes. Explain why two model totals on different covered cohorts cannot be ranked as if they measured the same facilities.

## Sources and evidence boundaries

Seeger & Minderhoud (2026), [*Sea level much higher than assumed in most coastal hazard assessments*](https://doi.org/10.1038/s41586-026-10196-1), with the [25 August 2026 addendum](https://doi.org/10.1038/s41586-026-11017-1). The pinned [October 2026 catalog, record 23341921](https://doi.org/10.5281/zenodo.23341921) describes corrected products and their licenses. Exact versions, URLs, roles and inspected metadata hashes are in `data/nature-sources.json`; current results are in `data/nature-aggregates.json`.

Original calculations/code here are independent MIT implementations. Corrected-raster derivatives are **CC BY-NC-SA 4.0**, distinct from the **CC BY 4.0** original lesson prose and paper. Cite the corrected product, underlying DEM and MDT/geoid sources. No private workbook, facility identifiers, names, coordinates or clinical counts are bundled. [Full source audit](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/source-audit.html) · [Data dictionary](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/data-dictionary.html) · [References](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/references.html).

In [5]:
print('Nature lesson complete: 15_nature_data_and_sampling')

Nature lesson complete: 15_nature_data_and_sampling
